# Controller T5 single selection · frozen 20261026
Private. 200 NQ + 200 FEVER, four arms + cached C1 (2000 records). Six active long-path engineering checks first; E unavailable, zero E model calls. No gold, development/final inputs, tuning, retry, scoring or promotion. Attach frozen code .bin and existing private T4 retrieval resources.

In [ ]:
from pathlib import Path
import hashlib,io,json,os,shutil,subprocess,sys,tarfile,zipfile
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
matches=list(Path('/kaggle/input').rglob('controller_t5_selection_v1.tar.gz.bin'))
assert len(matches)==1 and sha(matches[0])=='151676a8c07008257086701c5dbced4fdc0b5e0f98f75f6e88caf98dcbae8f6f', 'Attach the one frozen private code dataset'
BASE=Path('/tmp/controller_t5_151676a8c070');BASE.mkdir(exist_ok=False)
with tarfile.open(matches[0]) as t:
    assert all(m.isfile() and (BASE/m.name).resolve().is_relative_to(BASE.resolve()) for m in t.getmembers())
    t.extractall(BASE,filter='data')
ROOT=BASE/'controller_t5_selection_v1'
for rel,h in json.loads((ROOT/'bundle_checksums.json').read_text()).items():assert sha(ROOT/rel)==h,rel
OUTPUT=Path('/kaggle/working/controller_t5_selection_v1_outputs');assert not OUTPUT.exists()
LOG=Path('/kaggle/working/controller_t5_selection_v1_run.log')
os.environ['OMP_NUM_THREADS']='4';os.environ['TOKENIZERS_PARALLELISM']='false'
os.chdir(ROOT)
def run(command):
    with LOG.open('a') as log:
        p=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:log.write(line);log.flush();print(line,end='',flush=True)
        if p.wait():raise RuntimeError('Frozen selection command failed: '+repr(command))
try:
    run([sys.executable,'-m','pip','install','-r','requirements-controller-e-v2a.txt'])
    run([sys.executable,'-m','unittest','tests.test_controller_integration','tests.test_evidence_v2a','-q'])
    run([sys.executable,'scripts/run_controller_t5_selection.py','--preflight','--output-dir',str(OUTPUT)])
    markers=[p for p in Path('/kaggle/input').rglob('resource_bundle_v1.json') if json.loads(p.read_text()).get('schema_version')=='controller_t4_resource_bundle_v1']
    assert len(markers)==1, 'Attach existing private Controller T4 retrieval resources dataset'
    resources=markers[0].parent;registry=json.loads(markers[0].read_text())
    expected={'data/processed/indexes/nq_train_bge_base_offsets.npy': {'bytes': 9793080, 'sha256': '3e819f87f04f6069e27f0ba12f89493834cbb597e2e0d2d599aee585b63d2e5b'}, 'data/processed/indexes/nq_train_bge_base_ids.jsonl': {'bytes': 37947689, 'sha256': '7d5d51f81ae3f89f4d7407b46b71f5c42804e418042e49c647435457f3715581'}, 'data/processed/indexes/nq_train_bge_base.index': {'bytes': 3760493613, 'sha256': 'cd94b5cd2d5fec4736ba16197e2431a87d347640cd1c79da6a6daa2cac9bfd5d'}, 'data/processed/retrieval/nq_train_corpus.jsonl': {'bytes': 782900757, 'sha256': 'c4ec2cc61aa091bd4cd6492b7cbdc53c8265395d9ac1b9b6307ac748daf6940d'}, 'data/processed/indexes/nq_train_bge_metadata.json': {'bytes': 4134, 'sha256': 'c6ed43b627833dd6592a964e5e03eebfd1fad3ba627f2c66a568ad94e7914fbd'}, 'artifacts/controller/nq_mapping_full_v1/validation.json': {'bytes': 3220, 'sha256': 'cdff67e6dcdab1d7751623402a99b04a1cf80285b50c30f5daff93e840668ff1'}, 'data/processed/controller_resources/fever_v1/fever_corpus.jsonl': {'bytes': 3522640169, 'sha256': '29dde612e13203ec37a566c2284d5e312eabc8e66277b972a6e643511fe65fd1'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq.index': {'bytes': 401921924, 'sha256': 'a000e7e994a4da47ec1d7af9371eab3c9e4872b2eeefb2f786cb102b4528046b'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_ids.jsonl': {'bytes': 213760540, 'sha256': '9bcecd97153871da40edd6ebdee8b149a976174cf929685f77f5a34deba2539a'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_meta.json': {'bytes': 601, 'sha256': 'f4146debafd05ff4b78d8788265c9df8ceb59462d40883beb6ef70eda6e6c2be'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_offsets.npy': {'bytes': 43168976, 'sha256': 'd764edc54e65a2f7d7255a2efbe7ec04c64b29758c15d965ee336a83202b485a'}, 'artifacts/controller/fever_resources_v1/import_validation.json': {'bytes': 2156, 'sha256': '3f7eb4d05dea50a07241d2a5a63f9392ddda6ca0209b239c191159dc896b3c64'}, 'artifacts/controller/fever_resources_v1/structure_validation.json': {'bytes': 3270, 'sha256': '49ffb987228e84a13ff85254983cea327114a4e898c2da530e4864e4b41a7d72'}, 'artifacts/controller/fever_provider_v1/query_replay.json': {'bytes': 8920, 'sha256': 'a576c7a3e12bf0a450f913431a87bf8d72726932cd08eab9b1e3c147955e634f'}}
    assert registry['files']==expected and registry['contains_gold'] is False
    for rel,info in expected.items():
        src=resources/rel;assert src.stat().st_size==info['bytes'] and sha(src)==info['sha256'],rel
        dst=ROOT/rel;dst.parent.mkdir(parents=True,exist_ok=True);assert not dst.exists();dst.symlink_to(src)
    print('Resources verified; selection only; E unavailable; candidate is NOT promoted',flush=True)
    run([sys.executable,'-u','scripts/run_controller_t5_selection.py','--output-dir',str(OUTPUT)])
finally:
    OUTPUT.mkdir(exist_ok=True)
    if LOG.exists():(OUTPUT/'run.log').write_bytes(LOG.read_bytes())
    (OUTPUT/'input_execution_manifest.json').write_bytes((ROOT/'execution_manifest.json').read_bytes())
    checks={str(p.relative_to(OUTPUT)):sha(p) for p in OUTPUT.rglob('*') if p.is_file()}
    (OUTPUT/'output_checksums.json').write_text(json.dumps(checks,indent=2))
    archive=Path('/kaggle/working/controller_t5_selection_v1_outputs.zip')
    with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED,strict_timestamps=False) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
    print('Preserved output:',archive,'SHA256:',sha(archive),flush=True)
